# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog / posts', 'url': 'https://edwarddonner.com/posts/'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 3 relevant links


{'links': [{'type': 'company page',
   'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'company page',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [12]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
MiniMaxAI/MiniMax-M2.7
Updated
1 day ago
•
143k
•
884
tencent/HY-Embodied-0.5
Updated
3 days ago
•
1.06k
•
771
Qwen/Qwen3.6-35B-A3B
Updated
2 days ago
•
453
zai-org/GLM-5.1
Updated
about 19 hours ago
•
94.4k
•
1.29k
google/gemma-4-31B-it
Updated
6 days ago
•
3.2M
•
1.98k
Browse 2M+ models
Spaces
Running
on
Zero
Agents
Featured
492
OmniVoice
🌍
492
High-quality voice cloning TTS for 600+ languages
Running
on
Zero
MCP
2.07k
Wan2.2 14B Preview
🐌
2.07k
generate a video from an image with a text prompt
Running
87
Bonsai 1-bit WebGPU
🌳
87

In [13]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [14]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [15]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nMiniMaxAI/MiniMax-M2.7\nUpdated\n1 day ago\n•\n143k\n•\n884\ntencent/HY-Embodied-0.5\nUpdated\n3 days ago\n•\n1.06k\n•\n772\nQwen/Qwen3.6-35B-A3B\nUpdated\n2 days ago\n•\n459\nzai-org/GLM-5.1\nUpdated\nabout 19 hours ago\n•\n94.4k\n•\n1.29k\ngoogle/gemma-4-31B-it\nUpdated\n6 days ago\n•\n3.2M\n•\n1.98k\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nAgents\nFeatured\n492\nOmniVoice\n🌍\n49

In [16]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [17]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is a vibrant AI community and collaboration platform dedicated to building the future of machine learning. It serves as a central hub where the global machine learning (ML) community can create, discover, share, and collaborate on open-source ML models, datasets, and applications.

With over 2 million models, 500,000+ datasets, and 1 million+ applications available, Hugging Face is the home of machine learning innovation, spanning multiple data modalities including text, image, video, audio, and even 3D.

---

## Our Platform

- **Models**: Access and contribute to a vast library of 2M+ ML models covering a wide range of AI applications.
- **Datasets**: Browse and share from 500,000+ datasets to drive your projects forward.
- **Spaces**: Deploy and showcase ML-powered applications hosted directly on the platform.
- **Collaboration & Hosting**: Host unlimited public models, datasets, and applications with tools that facilitate teamwork and speed up ML workflows.
- **Enterprise Solutions**: Advanced compute and collaboration tools tailored for teams and businesses, helping enterprises accelerate AI innovation.
- **Open-Source Stack**: Leverage a fast, reliable open-source stack that empowers building and deploying models efficiently.

Explore AI Apps or browse models, datasets, and applications developed by the community worldwide.

---

## Community & Culture

Hugging Face stands out as a community-driven platform that cultivates learning, collaboration, and open sharing to democratize AI development. The culture is deeply rooted in:

- **Openness & Ethics**: Promoting an open and ethical AI future.
- **Collaboration**: Enabling machine learning engineers, scientists, and users to connect and build together.
- **Innovation**: Supporting fast iteration and cutting-edge research.
- **Inclusivity**: Welcoming contributors from all backgrounds to share their work and grow their portfolios.
- **Global Reach**: Supporting projects such as voice cloning for 600+ languages and video generation from images.

---

## Customers & Users

Hugging Face serves a diverse audience including:

- **Researchers & Scientists** - Access the latest models and datasets to power their work.
- **Machine Learning Engineers & Developers** - Build, deploy, and ship ML applications quickly.
- **Enterprises & Teams** - Utilize enterprise-grade solutions for collaboration, scale, and compute.
- **Educators & Students** - Learn from and contribute to a vast repository of ML resources.
- **AI Enthusiasts & Innovators** - Experiment with open-source tools and join the AI revolution.

---

## Careers at Hugging Face

Join a passionate team shaping the future of AI and machine learning!

At Hugging Face, you will work in an open, collaborative, and innovative environment that values creativity and the exchange of ideas. The company is continuously growing and looking for talented individuals passionate about AI, software engineering, research, community building, and enterprise solutions.

You will have opportunities to:

- Build tools used by millions worldwide.
- Collaborate with a global community of AI experts.
- Work on cutting-edge AI technologies and open-source projects.
- Contribute to building an ethical, open AI ecosystem.

Keep an eye on their careers page to apply for current openings and become part of the future of AI.

---

## Contact & Get Started

- **Website**: [huggingface.co](https://huggingface.co)
- **Sign Up**: Join the Hugging Face community to start exploring and contributing.
- **Enterprise Solutions**: Request a demo or learn more about team plans to accelerate your AI projects.

Become part of the AI movement with Hugging Face — where machine learning innovates and collaborates to transform the world.

---

## Brand Highlights

- Signature colors: Bright yellow (#FFD21E), vibrant orange (#FF9D00), modern grey (#6B7280).
- Friendly & open logo reflecting the community spirit.
- The brand embodies openness, collaboration, and technical excellence.

---

Build, learn, and grow with the platform powering the next generation of AI.

**Hugging Face – The AI community building the future.**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [18]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [19]:
stream_brochure("Edward Donner", "https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 6 relevant links


# Edward Donner: Innovating AI to Unlock Human Potential

---

## Who We Are

Edward Donner is the online hub of Ed Donner—a passionate AI engineer, entrepreneur, and educator dedicated to exploring and advancing Large Language Models (LLMs) and AI technologies. Ed is co-founder and CTO of **Nebula.io**, a company revolutionizing talent sourcing and management using advanced AI, including proprietary, patented models that match people with job roles more accurately and rapidly than traditional methods.

With a rich background including founding the AI startup *untapt* (acquired in 2021), Ed's work centers on applying AI for profound, positive real-world impact by helping people discover their true potential and pursue fulfilling careers aligned with the Japanese concept of *Ikigai*—their reason for being.

---

## Our Mission & Vision

- **Mission:** To empower individuals and organizations through AI-driven solutions that enable better career matches, increased workplace engagement, and personal fulfillment.
- **Vision:** To raise global human prosperity by connecting people to roles where they will thrive, inspired by the belief that 77% of workers currently lack inspiration at work—a gap AI can bridge.

---

## What We Offer

### AI Curriculum & Learning Resources

Ed leverages his expertise by creating top-rated AI courses available on Udemy, with over **500,000 students across 194 countries**. These best-sellers cover topics such as:

- AI Coding: From basic vibes to agentic engineering
- AI Builder with n8n: Creating agents and voice agents
- AI Engineering MLOps: Deploying AI to production

Learners benefit from practical, cutting-edge knowledge that transforms curious coders into proficient AI engineers.

### Innovative Products

- **Nebula.io Platform:** Uses generative AI and patented algorithms for talent acquisition helping recruiters source and engage top candidates without relying on traditional keyword searches.
- **Outsmart:** A platform that pits LLMs against each other in battles of diplomacy and deviousness, showcasing AI capabilities in creative problem-solving.
- **Connect Four Game:** A fun application of AI for strategic gameplay and learning.

---

## Company Culture

- Passionate about cutting-edge AI and fostering a community of like-minded enthusiasts.
- Encourages continuous learning and knowledge sharing.
- Values creativity and exploration—evident in Ed’s personal interests like electronic music production.
- Committed to real-world impact and ethical AI applications.
- Open and approachable, with Ed’s candid talks and courses extending learning beyond traditional classrooms.

---

## Join Us

Edward Donner and Nebula.io are the ideal place for:

- **AI Researchers and Engineers** who want to build state-of-the-art AI systems addressing real challenges.
- **Recruitment and HR Professionals** seeking innovative, AI-powered talent acquisition tools.
- **Learners and Innovators** eager to master AI technologies through comprehensive courses.
- **Collaborators and Investors** passionate about transforming the future of work and human potential.

---

## Connect with Edward Donner

- Website: [www.edwarddonner.com](http://www.edwarddonner.com)  
- Email: ed [at] edwarddonner [dot] com  
- Social Media:  
  - [LinkedIn](https://www.linkedin.com/in/edwarddonner)  
  - [Twitter](https://twitter.com/edwyn)  
  - [Facebook](https://www.facebook.com/edwarddonner)

---

Unlock your potential and join us in reshaping the future of work through AI innovation. Whether you want to learn, collaborate, or invest, Edward Donner offers the expertise and tools to propel you forward.

In [20]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("Sparky", "https://sparky0520.github.io")

Selecting relevant links for https://sparky0520.github.io by calling gpt-5-nano
Found 4 relevant links


# Sparky: Innovative AI & Systems Solutions

---

## About Sparky

Sparky is a cutting-edge technology company at the intersection of Artificial Intelligence, systems engineering, and creative ideas that are ready to ship. Founded by Darsh Jain, a passionate builder and innovator from New Delhi, Sparky leverages advanced AI techniques such as large language models (LLMs), Machine Learning (ML), and Reinforcement Learning (RL) to develop practical and scalable solutions, particularly in education, high-frequency trading, and productivity.

---

## Core Projects & Products

- **Aki - Japanese Learning Platform**  
  Aki is an AI-driven Japanese language learning platform tailored for multilingual India, designed especially for enterprise users. The platform focuses on delivering education in the learner’s mother tongue, making it accessible and effective.  
  [Learn more at learnjp.org](https://learnjp.org)

- **WPL Prediction Model**  
  A custom Machine Learning pipeline that accurately predicted 9 out of 11 matches in the 2026 Women’s Premier League (WPL), showcasing Sparky’s expertise in sports analytics and predictive modeling.

- **Day Trading Bot with Deep Reinforcement Learning**  
  A trading bot trained on extensive Bitcoin data, using a combination of multilayer perceptrons and PPO architectures to optimize decisions in volatile crypto markets.

- **Meridian Pomodoro Timer App**  
  A mobile productivity app with a travel theme, offering unique workflow customization features to help users achieve focused work sessions.

---

## Expertise & Technology Stack

- Programming Languages: C++, Python, JavaScript  
- Frameworks & Tools: React, React Native, AI Orchestration, LLM Tooling  
- Systems: High-frequency trading infrastructure, REST, WebSocket, FIX protocols  
- AI Competencies: Machine Learning, Deep Learning, Large Language Models, Voice Agents  
- Platforms: Proprietary AI APIs including OpenAI Realtime API and LiveKit  
- Other Skills: Arch Linux, Japanese language proficiency (N4 level)

---

## Company Culture

Sparky embodies a culture of innovation, continuous learning, and practical application of emerging technologies. The team values:

- Cross-disciplinary knowledge blending AI, systems engineering, and business  
- Building impactful solutions aimed at real-world problems  
- Encouraging creativity and experimentation in cutting-edge tech areas like AI and reinforcement learning  
- Remote flexibility with a global outlook, while rooted in India’s vibrant tech ecosystem

---

## Customers & Markets

Sparky currently focuses on:

- **Enterprises in India**: Providing tailored AI-powered education platforms that cater to multilingual workforces.  
- **Financial Services**: Building robust, automated trading infrastructures and market prediction models.  
- **Individual Productivity Enthusiasts**: Offering innovative apps to enhance personal effectiveness and workflows.

---

## Careers @ Sparky

If you're passionate about AI, ML, systems engineering, and solving complex challenges, Sparky is the place to grow your career. Ideal candidates are developers and engineers who:

- Are skilled in C++, Python, JavaScript, and modern AI frameworks  
- Thrive in fast-paced, startup-like environments where innovation meets execution  
- Want hands-on experience with next-gen AI tools including voice agents and LLM orchestration  
- Have an interest in education technologies, financial tech, or productivity applications

To explore opportunities, connect via LinkedIn or reach out directly to darshjain05.in@gmail.com.

---

## Contact

**Founder:** Darsh Jain  
**Location:** New Delhi, India  
**Email:** darshjain05.in@gmail.com  
**GitHub:** [github.com/sparky0520](https://github.com/sparky0520)  
**LinkedIn:** [linkedin.com/in/darsh-jain-521691218](https://linkedin.com/in/darsh-jain-521691218)

---

Sparky is committed to transforming ideas into impactful technologies with precision, creativity, and purpose. Join us on our journey to build the future of AI-enabled systems.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>